# 00 - Bienvenida al Clúster Spark

Este notebook comprueba que tu clúster está funcionando y te muestra el flujo básico:
1. Conectar al clúster (1 master + 2 workers)
2. Cargar datos
3. Procesarlos con Spark SQL / DataFrames
4. Entrenar un modelo de ML con MLlib

**Spark Master UI:** abre el puerto 8080 forwarded en Codespaces para ver el clúster en vivo.
**Spark Job UI:** puerto 4040, aparece cuando lanzas un job.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("clase-spark")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

spark

## Comprobación rápida: ¿cuántos workers ve el master?

In [ ]:
sc = spark.sparkContext
print("Master:", sc.master)
print("Núcleos por defecto:", sc.defaultParallelism)

## Ejemplo: cargar un CSV y procesarlo

In [ ]:
import pandas as pd
import numpy as np

# Genera un dataset de ejemplo si no existe
import os
if not os.path.exists("data/ejemplo.csv"):
    np.random.seed(42)
    n = 5000
    df_pd = pd.DataFrame({
        "edad": np.random.randint(18, 70, n),
        "ingresos": np.random.normal(30000, 8000, n).round(2),
        "horas_estudio": np.random.uniform(0, 10, n).round(2),
    })
    df_pd["aprueba"] = (
        (df_pd["horas_estudio"] * 5 + df_pd["ingresos"] / 10000 + np.random.normal(0, 3, n)) > 15
    ).astype(int)
    os.makedirs("data", exist_ok=True)
    df_pd.to_csv("data/ejemplo.csv", index=False)

df = spark.read.csv("data/ejemplo.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

## Entrenamiento de un modelo con MLlib (distribuido en el clúster)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

assembler = VectorAssembler(
    inputCols=["edad", "ingresos", "horas_estudio"],
    outputCol="features"
)
data = assembler.transform(df).select("features", "aprueba")

train, test = data.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="aprueba")
modelo = lr.fit(train)

predicciones = modelo.transform(test)
evaluador = BinaryClassificationEvaluator(labelCol="aprueba")
auc = evaluador.evaluate(predicciones)
print(f"AUC del modelo: {auc:.4f}")

Si ves un AUC > 0.5 y el job ha aparecido en el Spark Job UI (puerto 4040), todo funciona correctamente. ¡Listo para trabajar!